# Simulation Based Inference to Remove Sampling Bias - Designing Household Studies


Model  with parameters `beta`, `delta`, `mu_inf_SC`, `mu_inf_SA`, `mu_inf_AI`, `mu_inf_AC`, `mu_inf_AA`, `mu_susc_C`, `mu_susc_A`.
Neural networks trained with fewer households to see the impact of the number of households (this is done in the configurator).

Simulation study with different number of households and different maximal follow-up times (no new simulation required for that).

In [ ]:
import os
os.environ['KERAS_BACKEND'] = 'jax'

import pickle
import itertools
from matplotlib import pyplot as plt
from tqdm import tqdm
from joblib import Parallel, delayed

import numpy as np
import pandas as pd
from scipy.stats import gamma
import math

import keras
import bayesflow as bf

from cmdstanpy import CmdStanModel
from helper_functions import normalize_household_data
#from network_config import load_model
#from helper_functions import shorten_follow_up_time

In [ ]:
job_array_id = int(os.environ.get('SLURM_ARRAY_TASK_ID', 0))
n_procs = int(os.environ.get('SLURM_CPUS_PER_TASK', 1))
batch_size = 64
iterations_per_epoch = 100

## Define model and prior

We fix some parameters such that the model becomes identifiable. We fix the parameters `alpha`, `mu_protect_acq`, `mu_protect_transm`. We checked whether we need to fix  `delta` as well.

In [ ]:
# fixed_parameters = {
#     'alpha': {
#         "alpha": 0.001,  # introduces too much variation in the PedCov
#         "mu_protect_acq": 0.8,  # not enough PedCov to estimate
#         "mu_protect_transm": 1.  # not enough PedCov to estimate
#     },
#     'omicron': {
#         "alpha": 0.002,  # introduces too much variation in the PedCov
#         "mu_protect_acq": 0.8,  # not enough PedCov to estimate
#         "mu_protect_transm": 0.8  # not enough PedCov to estimate
#     }
# }
param_names = {
    'alpha': r'$\alpha$', 'beta': r'$\beta$', 'delta': r'$\delta$',
    'mu_inf_SC': r'$\mu_\text{inf}^\text{SC}$', 'mu_inf_SI': r'$\mu_\text{inf}^\text{SI}$',
    'mu_inf_AI': r'$\mu_\text{inf}^\text{AI}$', 'mu_inf_AC': r'$\mu_\text{inf}^\text{AC}$', 'mu_inf_AA': r'$\mu_\text{inf}^\text{AA}$',
    'mu_susc_C': r'$\mu_\text{susc}^\text{C}$', 'mu_susc_I': r'$\mu_\text{susc}^\text{I}$',
    #'mu_protect_acq': r'$\mu_\text{protect}^\text{acq}$', 'mu_protect_transm': r'$\mu_\text{protect}^\text{transm}$'
}

In [ ]:
# define the prior
def meta() -> dict:
    return dict(
        variant='alpha',  # alpha, omicron
        variant_id=0,  # 0 for alpha, 1 for omicron
        selection_procedure='random',  # pedcov, adult, random
        selection_procedure_id=0  # 0 for pedcov, 1 for adult, 2 for random
    )

def prior() -> dict:
    var = 1.0 # todo: 1
    params = {
        'alpha': np.random.uniform(0, 0.1) if 'alpha' in param_names.keys() else 0.001,
        'beta': np.random.uniform(0, 2) if 'beta' in param_names.keys() else 0.3,
        'delta': np.random.uniform(-3, 3) if 'delta' in param_names.keys() else 0.1,
        'mu_inf_SC': np.exp(np.random.normal(0, var)) if 'mu_inf_SC' in param_names.keys() else 1.0,
        'mu_inf_SI': np.exp(np.random.normal(0, var)) if 'mu_inf_SI' in param_names.keys() else 1.0,
        'mu_inf_AI': np.exp(np.random.normal(0, var)) if 'mu_inf_AI' in param_names.keys() else 1.0,
        'mu_inf_AC': np.exp(np.random.normal(0, var)) if 'mu_inf_AC' in param_names.keys() else 1.0,
        'mu_inf_AA': np.exp(np.random.normal(0, var)) if 'mu_inf_AA' in param_names.keys() else 1.0,
        'mu_susc_C': np.exp(np.random.normal(0, var)) if 'mu_susc_C' in param_names.keys() else 1.0,
        'mu_susc_I': np.exp(np.random.normal(0, var)) if 'mu_susc_I' in param_names.keys() else 1.0,
        'mu_protect_acq': np.exp(np.random.normal(0, var)) if 'mu_protect_acq' in param_names.keys() else 0.8,
        'mu_protect_transm': np.exp(np.random.normal(0, var)) if 'mu_protect_transm' in param_names.keys() else 1.0
    }
    return params

prior()

In [ ]:
def inv_logit(p):
    return np.exp(p) / (1 + np.exp(p))

# -----------------------------------------------------------------------------
# Precompute discretized generation‐time PDF
# -----------------------------------------------------------------------------
def get_generation_time(shapeInf, scaleInf, maxT=1000):
    """
    Returns a length-(maxT+1) array gpdf where
    gpdf[t] = P(generation time ∈ (t-1, t]] / (1 - P(T=0))
    """
    t = np.arange(0, maxT + 1)
    cdf = gamma.cdf(t, a=shapeInf, scale=scaleInf)
    # PDF mass on each unit interval
    pdf = np.diff(cdf, prepend=0)
    # conditional on T>=1
    pdf = pdf / (1 - pdf[0])
    return pdf


# -----------------------------------------------------------------------------
# Simulate Outbreak
# -----------------------------------------------------------------------------
def simulate_outbreak(hh_size, duration_followup,
                      age_cat, protected,
                      p_asympto, minIncub, maxIncub,
                      shapeIncub, scaleIncub,
                      gpdf, delayDist,
                      alpha, beta, delta,
                      mu_inf, mu_susc, mu_protect):
    """
    hh_size            : number of individuals in household
    duration_followup  : integer follow‐up length
    age_cat            : int[hh_size] (0=infant,1=child,2=adult)
    protected          : int[hh_size] (0/1)
    p_asympto          : float
    minIncub,maxIncub  : ints
    shapeIncub,scaleIncub : floats
    gpdf               : 1D array of generation‐time probabilities
    delayDist          : float[6] (shape1, rate1, shift1, shape2, rate2, shift2)
    alpha,beta,delta   : floats
    mu_inf             : float[5]
    mu_susc            : float[2]
    mu_protect         : float[2]
    """
    # Initialize state arrays
    inf_date      = -np.ones(hh_size, dtype=np.int64)
    infect_status = np.zeros(hh_size, dtype=np.int64)
    date_sympt    = np.full(hh_size, 1000, dtype=np.int64)
    is_index      = np.zeros(hh_size, dtype=np.int64)
    is_inclu      = np.zeros(hh_size, dtype=np.int64)
    incl_dt       = -np.ones(hh_size, dtype=np.float64)

    # Select index
    idx = np.random.randint(0, hh_size)
    is_index[idx] = 1
    inf_date[idx] = 0

    # Determine status & symptom time for index
    if np.random.rand() < p_asympto:
        infect_status[idx] = 2
        incub = np.random.randint(minIncub, maxIncub + 1)
    else:
        infect_status[idx] = 1
        incub = int(np.round(np.random.gamma(shapeIncub, scaleIncub)))
    date_sympt[idx] = incub

    # Propagate infection
    susceptibles = [i for i in range(hh_size) if i != idx]
    for t in range(1, duration_followup + 1):
        new_susc = []
        for ind in susceptibles:
            # Compute hazard at time t
            beta_foyer = 0.0
            for j in range(hh_size):
                if inf_date[j] >= 0 and 1 <= t - inf_date[j] < gpdf.shape[0]:
                    base = gpdf[t - inf_date[j]]
                    # map (age_cat[j], infect_status[j]) → idx_mu
                    if   age_cat[j] == 1 and infect_status[j] == 1: mu_idx = 0
                    elif age_cat[j] == 0 and infect_status[j] == 1: mu_idx = 1
                    elif age_cat[j] == 0 and infect_status[j] == 2: mu_idx = 2
                    elif age_cat[j] == 1 and infect_status[j] == 2: mu_idx = 3
                    else: mu_idx = 4
                    contrib = base * mu_inf[mu_idx]
                    if protected[j] == 1:
                        contrib *= mu_protect[1]
                    beta_foyer += contrib
            # susceptibility scaling
            if age_cat[ind] == 1:
                beta_foyer *= mu_susc[0]
            elif age_cat[ind] == 0:
                beta_foyer *= mu_susc[1]
            if protected[ind] == 1:
                beta_foyer *= mu_protect[0]
            # total hazard
            hazard = alpha + beta / ((hh_size / 4.0) ** delta) * beta_foyer  # convert to probability, not done by Sophie
            # p_t = 1- np.exp(-\int_{t-1}^{t} \lambda_i(s)\,ds) approx 1 - e^{-\lambda_i(t)}
            # the approximation is exact if \lambda_i is constant over the day
            p_t = 1 - np.exp(-hazard)  # discrete-time survival and hazard probabilities

            # infection event
            if np.random.rand() < p_t:
                inf_date[ind] = t
                if np.random.rand() < p_asympto:
                    infect_status[ind] = 2
                    incub = np.random.randint(minIncub, maxIncub + 1)
                else:
                    infect_status[ind] = 1
                    incub = int(np.round(np.random.gamma(shapeIncub, scaleIncub)))
                date_sympt[ind] = t + incub
            else:
                new_susc.append(ind)
        susceptibles = new_susc
        if len(susceptibles) == 0:
            break

    # Determine inclusion case & inclusion date
    infected = [l for l in range(hh_size) if infect_status[l] != 0]
    if len(infected) == 1:
        inc = infected[0]
    else:
        inc = np.random.choice(np.array(infected))
    is_inclu[inc] = 1
    # choose delay based on status
    if infect_status[inc] == 1:
        shape_d, rate_d, shift_d = delayDist[0], delayDist[1], delayDist[2]
    else:
        shape_d, rate_d, shift_d = delayDist[3], delayDist[4], delayDist[5]
    # note: numpy gamma uses scale = 1/rate
    delay = np.random.gamma(shape_d, 1.0 / rate_d) - shift_d
    incl_dt[inc] = date_sympt[inc] + delay

    # Re‐anchor dates so that min(date_sympt, inf_date, incl_dt) → t0=30
    all_dates = []
    for i in range(hh_size):
        if infect_status[i] != 0:
            all_dates.append(date_sympt[i])
        if inf_date[i] >= 0:
            all_dates.append(inf_date[i])
        if is_inclu[i] == 1:
            all_dates.append(int(incl_dt[i]))
    t0 = np.min(all_dates)
    for i in range(hh_size):
        if date_sympt[i] < 1000:
            date_sympt[i] = 30 + (date_sympt[i] - t0)
        if inf_date[i] >= 0:
            inf_date[i] = 30 + (inf_date[i] - t0)
        if is_inclu[i] == 1:
            incl_dt[i] = 30 + (incl_dt[i] - t0)
    end_follow_up = 30 + duration_followup - t0
    #end_follow_up = duration_followup
    return inf_date, date_sympt, infect_status, is_index, is_inclu, incl_dt, end_follow_up


def data_selection(d, variant, method):
    """
    Select households from simulation PedCov based on variant and method

    Parameters:
    -----------
    d : pd.DataFrame
        Household simulation PedCov
    variant : str
        Variant type ("alpha" or "omicron")
    method : str
        Selection method ("random", "pedcov", "adultcov")
    verbose : bool
        Whether to print debug messages

    Returns:
    --------
    pd.DataFrame : Selected and randomized household PedCov
    """
    d = d.copy()

    # Determine counts of households based on variant
    if variant == "alpha":
        tot_hh = 128  # 84
    elif variant == "omicron":
        tot_hh = 54   # 46
    else:
        raise ValueError("Variant must be 'alpha' or 'omicron'")

    tot_hh_a = math.ceil(tot_hh / 5)  # 1/5 of hh are included through asympto children
    tot_hh_s = tot_hh - tot_hh_a      # 4/5 of hh are included through sympto children

    if variant == "alpha":
        tot_hh_inclIndex = 88  # 49
    elif variant == "omicron":
        tot_hh_inclIndex = 44  # 36
    else:
        raise ValueError("Variant must be 'alpha' or 'omicron'")

    # Available households
    hh = list(d['id_hh'].unique())
    hh_origin = list(d['id_hh_origin'].unique())

    if method == "random":
        # Random sample from all households
        sel = np.random.choice(hh, size=min(tot_hh, len(hh)), replace=False)
        recruit = d[d['id_hh'].isin(sel)].copy()

    else:  # method in ["pedcov", "adultcov"]
        recruit = pd.DataFrame()

        hh_s = 0  # marker for count of sympto category
        hh_a = 0  # marker for count of asympto category
        hh_inclIndex = 0  # marker for count of inclusion=index category
        hh_inclNotIndex = 0  # marker for count of inclusion!=index category
        not_selected = 0

        while hh_s + hh_a < tot_hh and len(hh) > 0:
            # While there is not enough hh in the final base
            u = np.random.choice(hh)  # pick one randomly
            u_origin = d[d['id_hh'] == u]['id_hh_origin'].iloc[0]  # placeholder

            # Get household PedCov
            hh_data = d[d['id_hh'] == u]
            inclusion_case = hh_data[hh_data['is_incluCase'] == 1]

            if len(inclusion_case) == 0:
                # No inclusion case found, skip
                not_selected += 1
                hh = [h for h in hh if h != u]
                continue

            inclusion_case = inclusion_case.iloc[0]

            if method == "pedcov":
                # If inclusion case is an adult --> exclude and go to next iteration
                if inclusion_case['age_exact'] > 18:
                    not_selected += 1
                    hh = [h for h in hh if h != u]
                    continue
            elif method == "adultcov":
                # If inclusion case is NOT an adult --> exclude and go to next iteration
                if inclusion_case['age_exact'] <= 18:
                    not_selected += 1
                    hh = [h for h in hh if h != u]
                    continue

            # If inclusion case has not had symptoms or test yet --> exclude
            if inclusion_case['date_sympt'] >= inclusion_case['incl_dt']:
                not_selected += 1
                hh = [h for h in hh if h != u]  # Remove this hh from the list of pickable hh
                continue

            # Check if inclusion case is also index case
            is_index = inclusion_case['is_index'] == 1
            infect_status = inclusion_case['infect_status']

            if is_index and hh_inclIndex < tot_hh_inclIndex:
                # Inclusion case is also index case (and the count for this type is not full)
                if infect_status == 1 and hh_s < tot_hh_s:
                    # If inclusion case is symptomatic (and the count for this type is not full)
                    recruit = pd.concat([recruit, hh_data], ignore_index=True)
                    hh_s += 1  # increase count of sympto incl cases
                    hh_inclIndex += 1  # increase count of incl case also index

                    hh_origin = [h for h in hh_origin if h != u_origin]  # Remove from pickable hh_origin

                elif infect_status == 2 and hh_a < tot_hh_a:
                    # If inclusion case is asymptomatic (and the count for this type is not full)
                    recruit = pd.concat([recruit, hh_data], ignore_index=True)
                    hh_a += 1  # increase count of asympto incl cases
                    hh_inclIndex += 1  # increase count of incl case also index

                    hh_origin = [h for h in hh_origin if h != u_origin]  # Remove from pickable hh_origin

                else:
                    not_selected += 1  # If the count for this type of sympto status is already full

            elif not is_index and hh_inclNotIndex < (tot_hh - tot_hh_inclIndex):
                # Inclusion case is not the index case (and the count for this type is not full)
                if infect_status == 1 and hh_s < tot_hh_s:
                    # If inclusion case is symptomatic (and the count for this type is not full)
                    recruit = pd.concat([recruit, hh_data], ignore_index=True)
                    hh_s += 1  # increase count of sympto incl cases
                    hh_inclNotIndex += 1  # increase count of incl case not index

                    hh_origin = [h for h in hh_origin if h != u_origin]  # Remove from pickable hh_origin

                elif infect_status == 2 and hh_a < tot_hh_a:
                    # If inclusion case is asymptomatic (and the count for this type is not full)
                    recruit = pd.concat([recruit, hh_data], ignore_index=True)
                    hh_a += 1  # increase count of asympto incl cases
                    hh_inclNotIndex += 1  # increase count of incl case not index

                    hh_origin = [h for h in hh_origin if h != u_origin]  # Remove from pickable hh_origin

                else:
                    not_selected += 1  # If the count for this type of sympto status is already full

            else:
                not_selected += 1  # If the count for this type of incl case / index is already full

            hh = [h for h in hh if h != u]  # Remove this hh from the list of pickable hh
    # Randomize order of households
    if len(recruit) > 0:
        unique_households = recruit['id_hh'].unique()
        random_order = np.random.permutation(len(unique_households))

        # Create mapping from household ID to random order
        hh_order_map = dict(zip(unique_households, random_order))

        # Add random order column and sort
        recruit['random_order'] = recruit['id_hh'].map(hh_order_map)
        randomized_recruit = recruit.sort_values('random_order').drop('random_order', axis=1).reset_index(drop=True)
    else:
        randomized_recruit = recruit

    return randomized_recruit

# -----------------------------------------------------------------------------
# Python wrapper to run across all households
# -----------------------------------------------------------------------------
class OutbreakSimulator:
    def __init__(self, variant, n_repeat=50):
        self.variant   = variant
        self.n_repeat  = n_repeat
        self.minimal_length = 9  # minimal length of the household PedCov -> time steps

        if variant == "alpha":
            self.p_asympto = 0.22
            mIncub, sdIncub = 4.42, 2.3
            self.minIncub, self.maxIncub = 2, 7
            self.shapeInf, self.scaleInf = 2, 1/0.44
            self.delayDist = [6.9368753, 0.7376425, -3.0000000, 5.6516107, 0.9719026, 2.0000000]
        elif variant == "omicron":
            self.p_asympto = 0.25
            mIncub, sdIncub = 3.09, 1.64
            self.minIncub, self.maxIncub = 1, 5
            self.shapeInf, self.scaleInf = 3.531, 1/1.098
            self.delayDist = [8.4310842, 0.9508425, -4.0000000, 3.8209443, 1.2737556, 1.0000000]
        else:
            raise ValueError(f"Unknown variant: {variant}")
        if self.delayDist[2] < 0:  # taken from sophie's code, but not sure why this is needed
            self.delayDist[2] = abs(self.delayDist[2]) + 1
        if self.delayDist[5] < 0:
            self.delayDist[5] = abs(self.delayDist[5]) + 1

        self.shapeIncub = mIncub**2 / sdIncub**2
        self.scaleIncub = sdIncub**2 / mIncub
        self.gpdf       = get_generation_time(self.shapeInf, self.scaleInf)

        # load PedCov
        self.df_raw = pd.read_table(f"Simulator/pedcovid_data_structure_{self.variant}.txt", delimiter=' ')

    def __call__(self,
                 alpha, beta, delta,
                 mu_inf_SC, mu_inf_SI, mu_inf_AI, mu_inf_AC, mu_inf_AA,
                 mu_susc_C, mu_susc_I,
                 mu_protect_acq, mu_protect_transm,
                 method="random",
                 return_df=False
                 ):
        if isinstance(method, str):
            method = [method]
        for m in method:
            if m not in ["random", "pedcov", "adultcov"]:
                raise ValueError("Method must be one of: 'random', 'pedcov', 'adultcov'")

        # Replicate the dataframe n_repeat times
        df = pd.concat([self.df_raw] * self.n_repeat, ignore_index=True)

        # Generate id_rep
        df["id_rep"] = df.groupby(["id_hh", "id_patient"]).cumcount() + 1

        # Recreate the id_patient
        df["id_patient"] = df.groupby("id_hh").cumcount() + 1  # Row number within id_hh

        # Generate the test column
        df["test"] = df.groupby(["id_hh", "id_rep"]).ngroup() + 1  # Group number (test)

        # Combine id_hh and id_rep
        df["id_hh"] = df["id_hh"].astype(str) + "-" + df["id_rep"].astype(str)

        # Extract the original id_hh
        df["id_hh_origin"] = df["id_hh"].str.extract(r"^([0-9]+)")

        # 2) for each household, run simulate_outbreak
        results = []
        for hh, hh_df in df.groupby("id_hh", sort=False):
            N = len(hh_df)
            # extract arrays
            duration = int(hh_df["duration_followup"].iat[0])
            age_cat  = hh_df["age"].astype(np.int64).values
            protected= hh_df["protected"].astype(np.int64).values

            out = simulate_outbreak(
                hh_size=N,
                duration_followup=duration,
                age_cat=age_cat,
                protected=protected,
                p_asympto=self.p_asympto,
                minIncub=self.minIncub,
                maxIncub=self.maxIncub,
                shapeIncub=self.shapeIncub,
                scaleIncub=self.scaleIncub,
                gpdf=self.gpdf,
                delayDist=self.delayDist,
                alpha=alpha,
                beta=beta,
                delta=delta,
                mu_inf=np.array([
                    mu_inf_SC,
                    mu_inf_SI,
                    mu_inf_AI,
                    mu_inf_AC,
                    mu_inf_AA
                ], dtype=np.float64),
                mu_susc=np.array([
                    mu_susc_C,
                    mu_susc_I
                ], dtype=np.float64),
                mu_protect=np.array([
                    mu_protect_acq,
                    mu_protect_transm
                ], dtype=np.float64)
            )

            # attach outputs back to DataFrame
            hh_df = hh_df.reset_index(drop=True)
            hh_df["inf_date"], hh_df["date_sympt"], hh_df["infect_status"], \
            hh_df["is_index"], hh_df["is_incluCase"], hh_df["incl_dt"], hh_df["end_followup"] = out

            results.append(hh_df)

        # 3) combine all households
        new_data_full = pd.concat(results, ignore_index=True)

        # 4) Selection of households
        data_list = []
        norm_data_list = []
        for m in method:
            new_data = data_selection(new_data_full, variant=self.variant, method=m)
            new_data['select_process'] = m
            new_data['variant'] = self.variant
            new_data = new_data[['id_patient', 'id_hh', 'id_hh_origin', 'hh_size', 'date_sympt',
                                 'infect_status', 'end_followup', 'age', 'age_exact', 'protected',
                                 'variant', 'select_process']]
            # add method to id_hh
            new_data['id_hh'] = new_data['id_hh'] + '-' + m

            data_list.append(new_data)
            # normalize the household PedCov for input to the neural network
            new_data_norm = normalize_household_data(new_data, minimal_length=self.minimal_length)
            norm_data_list.append(new_data_norm)
        data_list = pd.concat(data_list, ignore_index=True)
        norm_data_list = np.stack(norm_data_list, axis=0)
        if len(method) == 1:
            norm_data_list = norm_data_list[0]  # remove first dimension if only one method
        if return_df:
            return dict(sim_data_df=data_list, sim_data=norm_data_list)
        else:
            return dict(sim_data=norm_data_list)
simulator = OutbreakSimulator("alpha")

simulator_bf = lambda alpha, beta, delta, mu_inf_SC, mu_inf_SI, mu_inf_AI, mu_inf_AC, mu_inf_AA, mu_susc_C, mu_susc_I, mu_protect_acq, mu_protect_transm, method='random': simulator(alpha, beta, delta,
                        mu_inf_SC, mu_inf_SI, mu_inf_AI, mu_inf_AC, mu_inf_AA,
                        mu_susc_C, mu_susc_I,
                        mu_protect_acq, mu_protect_transm,
                        method)

In [ ]:
%%time
test_params = prior()
test = simulator(**test_params, return_df=True, method='random')
test['sim_data_df'].head()

In [ ]:
test['sim_data_df'].infect_status.value_counts()

In [ ]:
generative_model = bf.make_simulator([meta, prior, simulator_bf]) #, meta_fn=meta)

In [ ]:
stan_model = CmdStanModel(stan_file='pedcov_stan_model.stan')
def get_stan_posterior(obs_df, max_t=1000, chains=4, show_progress=False):
    """
    Prepare PedCov and fit the household infection model

    Parameters:
    obs_df: DataFrame
    max_t: maximum follow-up time
    chains: number of MCMC chains
    """

    # Sort by household and patient ID for consistent indexing
    df_sorted = obs_df.sort_values(['id_hh', 'id_patient']).reset_index(drop=True)

    # Create mappings
    N = len(df_sorted)
    unique_hh = df_sorted['id_hh'].unique()
    H = len(unique_hh)

    # Map household IDs to consecutive integers 1, 2, ..., H
    hh_id_map = {hh_id: idx + 1 for idx, hh_id in enumerate(unique_hh)}
    hh_id_array = [hh_id_map[hh_id] for hh_id in df_sorted['id_hh']]

    # Get household sizes
    hh_sizes = df_sorted.groupby('id_hh')['id_patient'].count()
    hh_size_array = [hh_sizes[hh_id] for hh_id in unique_hh]

    # Convert other variables
    end_time = df_sorted['end_followup'].astype(int).tolist()
    obs_time = df_sorted['date_sympt'].astype(int).tolist()
    age_cat = df_sorted['age'].astype(int).tolist()
    infect_status = df_sorted['infect_status'].astype(int).tolist()
    protected = df_sorted['protected'].astype(int).tolist()

    # Prepare Stan PedCov
    stan_data = {
        'N': N,
        'H': H,
        'maxT': max_t,
        'hh_id': hh_id_array,
        'hh_size': hh_size_array,
        'end_time': end_time,
        'infect_status': infect_status,
        'obs_time': obs_time,
        'age_cat': age_cat,
        'is_protected': protected,
        'kt_shape': simulator.shapeInf,
        'kt_scale': simulator.scaleInf,
        'inc_shape_symp': simulator.delayDist[0],
        'inc_rate_symp': simulator.delayDist[1],
        'inc_shift_symp': simulator.delayDist[2],
        'inc_shape_asymp': simulator.delayDist[3],
        'inc_rate_asymp': simulator.delayDist[4],
        'inc_shift_asymp': simulator.delayDist[5],
    }

    if show_progress:
        # Print PedCov summary for debugging
        print(f"Data summary:")
        print(f"  N (individuals): {N}")
        print(f"  H (households): {H}")
        print(f"  Max household size: {max(hh_size_array)}")
        print(f"  Infections observed: {sum(inf > 1 for inf in infect_status)}")
        print(f"  Protected individuals: {sum(protected)}")

    # Fit the model to the PedCov
    fit = stan_model.sample(
        data=stan_data,
        show_progress=show_progress,
        chains=chains,
    )

    # Extract posterior samples
    param_samples = {}
    for param in param_names:
        param_samples[param] = fit.draws_pd(param)
    return param_samples

In [ ]:
def list_of_dicts_to_dict_of_lists(list_of_dicts):
    # Check if the list is empty
    if not list_of_dicts:
        return {}

    # Initialize the dictionary of arrays
    dict_of_arrays = {key: [] for key in list_of_dicts[0]}

    # Populate the dictionary of arrays
    for dictionary in list_of_dicts:
        for key, value in dictionary.items():
            dict_of_arrays[key].append(value)
    for k in dict_of_arrays:
        dict_of_arrays[k] = np.array(dict_of_arrays[k])
    return dict_of_arrays

In [ ]:
list_target = []
list_estimate = []
for i in range(2):
    test_params = prior()
    test_data = simulator(**test_params, return_df=True)
    test_stan = get_stan_posterior(test_data['sim_data_df'], show_progress=True)

    list_target.append(test_params)
    list_estimate.append(test_stan)

list_target = list_of_dicts_to_dict_of_lists(list_target)
list_estimate = list_of_dicts_to_dict_of_lists(list_estimate)

In [ ]:
bf.diagnostics.recovery(list_estimate, list_target, variable_names=list(param_names.values()));

In [ ]:
param_names.values()

In [ ]:
list_estimate

In [ ]:
np.exp(np.median(list_estimate['alpha'], axis=1))

## Generate training data

In [ ]:
batch_size = 32
num_training_batches = 500
num_validation_sets = 10
training_data_file = f'models/training_data_pedcov.pickle'
validation_data_file = f'models/valid_data_pedcov.pickle'

In [ ]:
@delayed
def simulate_batch() -> dict:
    return generative_model.sample(batch_size)

In [ ]:
if os.path.exists(validation_data_file):
    # load simulation PedCov
    with open(validation_data_file, 'rb') as f:
        validation_data = pickle.load(f)
    try:
        with open(training_data_file, 'rb') as f:
            training_data = pickle.load(f)
    except FileNotFoundError:
        pass
else:
    training_data = Parallel(n_jobs=-1, verbose=1)(simulate_batch() for _ in range(num_training_batches))
    training_data = list_of_dicts_to_dict_of_lists(training_data)

    validation_data = Parallel(n_jobs=-1, verbose=1)(simulate_batch() for _ in range(num_validation_sets))
    validation_data = list_of_dicts_to_dict_of_lists(validation_data)

    for k in training_data.keys():
        training_data[k] = training_data[k].reshape(batch_size*num_training_batches, *training_data[k].shape[2:])
    for k in validation_data.keys():
        validation_data[k] = validation_data[k].reshape(batch_size*num_validation_sets, *validation_data[k].shape[2:])

    # pickle
    #with open(training_data_file, 'wb') as f:
    #    pickle.dump(training_data, f)
    #with open(validation_data_file, 'wb') as f:
    #    pickle.dump(validation_data, f)
    #exit()

## Neural Posterior Estimation

In [ ]:
adapter = (
    bf.adapters.Adapter()
    .drop(['variant', 'selection_procedure'])
    .to_array()
    .convert_dtype(from_dtype="float64", to_dtype="float32")
    .constrain(param_names, lower=0, inclusive='none', method="exp")
    .concatenate(param_names, into="inference_variables")
    .concatenate(['variant_id', 'selection_procedure_id'], into="inference_conditions")
    .rename('sim_data', to_key="summary_variables")
    .standardize('inference_variables')
)

In [ ]:
from bayesflow.utils.serialization import serializable

@serializable("bayesflow.networks")
class DoubleSummaryNetwork(bf.networks.SummaryNetwork):
    def __init__(self, inner_network, outer_network, name=None, **kwargs):
        super().__init__(**kwargs)
        self.name = 'inner_' + inner_network.name + '_outer_' + outer_network.name if name is None else name
        self.inner_network = inner_network  # operates over elements
        self.outer_network = outer_network  # operates over observations

    def call(self, x, training: bool = False, **kwargs):
        b_size, n_outer_obs, n_inner_obs = keras.ops.shape(x)[:3]

        # Flatten to combine batch and outer observation dimensions
        x_flat = keras.ops.reshape(x, (b_size * n_outer_obs, n_inner_obs, *keras.ops.shape(x)[3:]))

        # Apply the inner network to each element in the outer observation
        inner_output = self.inner_network(x_flat, training=training, **kwargs)

        # Reshape back to (b_size, n_outer_obs, inner_output_dim)
        inner_output = keras.ops.reshape(inner_output, (b_size, n_outer_obs, *keras.ops.shape(inner_output)[1:]))

        # Apply the outer network to the inner outputs
        outer_output = self.outer_network(inner_output, training=training, **kwargs)
        return outer_output


    def get_config(self):
        config = super().get_config()
        config.update({
            "inner_network": self.inner_network,
            "outer_network": self.outer_network
        })
        return config

In [ ]:
epochs = 100
inf_i, in_summary_dim, out_summary_dim = list(itertools.product([0, 1], [4, 8, 16], [4, 8, 16]))[job_array_id]
summary_network = DoubleSummaryNetwork(
         inner_network=bf.networks.TimeSeriesNetwork(summary_dim=in_summary_dim),
         outer_network=bf.networks.DeepSet(summary_dim=out_summary_dim),
         name=f'double_deep_set'
)
inference_network = [bf.networks.CouplingFlow(),
                     #bf.networks.ConsistencyModel(epochs*num_training_batches*batch_size),
                    bf.networks.FlowMatching()][inf_i]

model_name = f'pedcov_double_deep_set_{"coupling_flow" if inf_i == 0 else "flow_matching"}_{in_summary_dim}_{out_summary_dim}.keras'

workflow = bf.BasicWorkflow(
    adapter=adapter,
    summary_network=summary_network,
    inference_network=inference_network
)

model_path = f'models/{model_name}'
print(model_path)
if os.path.exists(model_path):
    workflow.approximator = keras.saving.load_model(filepath=model_path)
else:
    history = workflow.fit_offline(
        data=training_data,
        epochs=epochs,
        batch_size=batch_size,
        validation_data=validation_data,
    )
    #workflow.approximator.save(model_path)

In [ ]:
diagnostics = workflow.plot_default_diagnostics(test_data=validation_data, num_samples=300,
                                                calibration_ecdf_kwargs={'difference': True})

## Simulation Study

All parameters are set to 1 besides one parameter. Inference workflow applied on the full model still. Check if for the changing parameter the true parameter is still recovered.

Here we tell the network the right selection procedure. Results should be not be biased for no selection procedure. 

In [ ]:
# generate simulation study
n_sims_per_param = 50
fixed_params = {
    'alpha': 0.001,
    'beta': 0.2,
    'delta': 1.3,
    'mu_protect_acq': 0.8,
    'mu_protect_transm': 1.0,
}
study_params = ['mu_inf_SC', 'mu_inf_SI', 'mu_inf_AI', 'mu_inf_AC', 'mu_inf_AA', 'mu_susc_C', 'mu_susc_I']
selection_procedures = ['pedcov', 'adultcov', 'random']

sim_study_data = []
sim_study_params = []
sim_study_params_stan = {
    'pedcov': [], 'adultcov': [], 'random': []
}
for study_p in study_params:
    params = fixed_params.copy()
    params.update({k: 1. for k in study_params})  # set all parameters to 1
    for _ in range(n_sims_per_param):
        new_param = prior()
        params[study_p] = new_param[study_p]  # set the i-th parameter to a new value

        # run the simulator
        sim_data = simulator(**params, return_df=True, method=selection_procedures)

        # save the simulation PedCov and parameters
        sim_study_data.append(sim_data['sim_data'])
        sim_study_params.append(params.copy())

        # get the posterior samples with STAN
        sim_data_df = sim_data['sim_data_df']
        for sp in selection_procedures:
            sim_data_df_temp = sim_data_df[sim_data_df['selection_process'] == sp]
            sim_study_stan = get_stan_posterior(sim_data_df_temp)
            sim_study_params_stan[sp].append(sim_study_stan)

sim_study_data = list_of_dicts_to_dict_of_lists(sim_study_data)
sim_study_data.update(list_of_dicts_to_dict_of_lists(sim_study_params))
for sp in selection_procedures:
    sim_study_params_stan[sp] = list_of_dicts_to_dict_of_lists(sim_study_params_stan[sp])

# save the PedCov
#with open(f'models/pedcov_sim_study.pickle', 'wb') as f:
#    pickle.dump(sim_study_data, f)
#with open(f'models/pedcov_sim_study_stan.pickle', 'wb') as f:
#    pickle.dump(sim_study_params_stan, f)

In [ ]:
sim_study_data.keys()

In [ ]:
# perform inference on simulation study
results_sim_study = {}
n_samples_study = 10  # increase for more accurate results, e.g. 1000 samples
keep_log_transform = False

for selection_data, selection_inference in tqdm(product(['pedcov', 'random'], ['pedcov', 'random']), total=4):
    for variant in ['alpha', 'omicron']:
        # estimate median and CI for each parameter
        results_sim_study_vs = {
            'posterior_median': np.ones((len(non_fix_indices_study), n_sims_per_param)) * np.nan,
            'posterior_CI': np.ones((len(non_fix_indices_study), 6, n_sims_per_param)) * np.nan,
            'relative_bias_median': np.ones((len(non_fix_indices_study), n_sims_per_param)) * np.nan,
            'relative_bias_CI': np.ones((len(non_fix_indices_study), 6, n_sims_per_param)) * np.nan,
            'overall_median_error': np.ones((len(non_fix_indices_study), n_sims_per_param)) * np.nan,
            'n_infected_rel': np.ones((len(non_fix_indices_study), n_sims_per_param)) * np.nan
        }
        for i, index_i in enumerate(non_fix_indices_study):
            test = sim_study_dict[f'{variant}_{selection_inference}']['sim_batchable_context'][i]
            # get PedCov for this parameter
            sim_study_i = {
                'sim_data': sim_study_dict[f'{variant}_{selection_data}']['sim_data'][i],
                'prior_draws': sim_study_dict[f'{variant}_{selection_data}']['prior_draws'][i],
                # on purpose a different selection procedure than the PedCov was generated with
                'sim_batchable_context': sim_study_dict[f'{variant}_{selection_inference}']['sim_batchable_context'][i],
                'sim_non_batchable_context': sim_study_dict[f'{variant}_{selection_inference}']['sim_non_batchable_context']
            }
            simulation_study_data_conf = trainer.configurator(sim_study_i)
            # get posterior samples
            posterior_samples_sim_study = (
                trainer.amortizer.sample(simulation_study_data_conf, n_samples=n_samples_study)
            )
            posterior_samples_sim_study = renormalize_params(posterior_samples_sim_study,
                                                             keep_log_transform=keep_log_transform)
            if isinstance(simulation_study_data_conf, list):  # for ensemble
                prior_draws_study = renormalize_params(simulation_study_data_conf[0]['parameters'],
                                                       keep_log_transform=keep_log_transform)
            else:
                prior_draws_study = renormalize_params(simulation_study_data_conf['parameters'],
                                                       keep_log_transform=keep_log_transform)

            # get median and CI for each parameter of posterior samples
            # we only care about the changing parameter, the others are set to 1
            results_sim_study_vs['posterior_median'][i] = np.median(posterior_samples_sim_study[:, :, index_i], axis=1)
            results_sim_study_vs['posterior_CI'][i] = np.quantile(posterior_samples_sim_study[:, :, index_i],
                                                                    [0.005, 0.025, 0.1,
                                                                     0.995, 0.975, 0.9],
                                                                    axis=1)
             # get median and CI for each parameter of relative bias (estimate - true) / true
            relative_bias = (posterior_samples_sim_study[:, :, index_i] - prior_draws_study[:, index_i][:, np.newaxis]) / prior_draws_study[:, index_i][:, np.newaxis]
            results_sim_study_vs['relative_bias_median'][i] = np.median(relative_bias, axis=1)
            results_sim_study_vs['relative_bias_CI'][i] = np.quantile(relative_bias,
                                                                        [0.005, 0.025, 0.1,
                                                                         0.995, 0.975, 0.9],
                                                                        axis=1)
            # get overall median error
            results_sim_study_vs['overall_median_error'][i] = np.sum(
                (np.median(posterior_samples_sim_study, axis=1) - prior_draws_study)**2, axis=1
            )

            # get number of infected people in that dataset
            results_sim_study_vs['n_infected_rel'][i] = percentage_infection_age(sim_study_i['sim_data'],
                                                                                 param_names[index_i])

        results_sim_study[f'{selection_data}_{selection_inference}_{variant}'] = results_sim_study_vs

In [ ]:
# plot the results
prior_transformed = renormalize_params(prior_draws_sim_study, keep_log_transform=keep_log_transform)

for selection_data, selection_inference in product(['pedcov', 'random'], ['pedcov', 'random']):
    fig, ax = plt.subplots(2, len(non_fix_indices_study), sharex=True, sharey=True, figsize=(10, 6), tight_layout=True)
    space = 1. / n_sims_per_param
    ci_level = 0.95
    ci_interval = {0.99: [0, 3], 0.95: [1, 4], 0.8: [2, 5]}[0.95]
    colors = ['#1f78b4', '#b2df8a']

    for i, index_i in enumerate(non_fix_indices_study):
        relative_alpha = results_sim_study[f'{selection_data}_{selection_inference}_alpha']['relative_bias_median'][i]
        relative_omicron = results_sim_study[f'{selection_data}_{selection_inference}_omicron']['relative_bias_median'][i]

        # compute the error of the CI
        alpha_err = np.abs(results_sim_study[f'{selection_data}_{selection_inference}_alpha']['relative_bias_CI'][i][ci_interval] -
                           results_sim_study[f'{selection_data}_{selection_inference}_alpha']['relative_bias_median'][i])
        omicron_err = np.abs(results_sim_study[f'{selection_data}_{selection_inference}_omicron']['relative_bias_CI'][i][ci_interval] -
                             results_sim_study[f'{selection_data}_{selection_inference}_omicron']['relative_bias_median'][i])


        plot_alpha = np.ones(prior_transformed[:, index_i].shape[0], dtype=bool)
        plot_omicron = np.ones(prior_transformed[:, index_i].shape[0], dtype=bool)
        n_infected_rel_a = results_sim_study[f'{selection_data}_{selection_inference}_alpha']['n_infected_rel'][i].copy()
        n_infected_rel_o = results_sim_study[f'{selection_data}_{selection_inference}_omicron']['n_infected_rel'][i].copy()
        #plot_alpha = n_infected_rel_a > np.median(n_infected_rel_a)
        #plot_omicron = n_infected_rel_o > np.median(n_infected_rel_o)
        #plot_alpha =  prior_transformed[:, index_i] > 0.5
        for j in range(relative_alpha.shape[0]):
            handle_alpha = ax[0, i].errorbar(prior_transformed[:, index_i][j],
                                             relative_alpha[j],
                                             yerr=alpha_err[:, j][:, np.newaxis],
                                             fmt='o', color=colors[0], alpha=1 if plot_alpha[j] else 0.1,
                                             label='Alpha')
            handle_omicron = ax[1, i].errorbar(prior_transformed[:, index_i][j],
                                               relative_omicron[j],
                                               yerr=omicron_err[:, j][:, np.newaxis],
                                               fmt='o', color=colors[1], alpha=1 if plot_omicron[j] else 0.1,
                                               label='Omicron')

        handle_alpha_mean = ax[0, i].errorbar(1.5,
                                              np.median(relative_alpha[plot_alpha]),
                                              yerr=np.std(relative_alpha[plot_alpha]),
                                              color='black', marker='x', label=r'Median bias ($\pm$ std)', zorder=4)

        handle_omicron_mean = ax[1, i].errorbar(1.5,
                                                np.median(relative_omicron[plot_omicron]),
                                                yerr=np.std(relative_omicron[plot_omicron]),
                                                color='black', marker='x', label=r'Median bias ($\pm$ std)', zorder=4)

        ax[0, i].axhline(0, color='black', ls='--', zorder=3)
        ax[1, i].axhline(0, color='black', ls='--', zorder=3)

        ax[1, i].set_xlabel(param_names[index_i])
        ax[1, i].set_xlim(-0.5, 3.5)
        ax[1, i].set_xticks(np.arange(4))
        ax[0, 0].set_ylim(-1.5, 8)
    ax[0, 0].set_ylabel(f'Relative Bias\n(Median, {round(ci_level*100)}% CI)')
    ax[1, 0].set_ylabel(f'Relative Bias\n(Median, {round(ci_level*100)}% CI)')
    ax[0, n_params//2-1].set_title(f'Simulation Study - Data: {selection_data}, Inference: {selection_inference}')
    fig.legend(handles=[handle_alpha, handle_omicron, handle_alpha_mean],
                loc='lower center', ncol=3, bbox_to_anchor=(0.5, -0.05))
    #fig.savefig(f'{results_folder}/{amortizer_name}_sim_study_data_'
    #            f'{selection_data}_vs_inference_{selection_inference}.png', bbox_inches='tight')
    plt.show()

In [ ]:
# plot bias in both scenarios
fig, ax = plt.subplots(2, len(non_fix_indices_study), sharex=True, sharey=True, figsize=(10, 6), tight_layout=True)
colors = ['#d95f02', '#7570b3']

for j, (selection_data, selection_inference) in enumerate(product(['pedcov', 'random'], ['pedcov', 'random'])):
    if selection_data != selection_inference:
        # using the wrong condition can be used as sanity check
        continue
    for i, index_i in enumerate(non_fix_indices_study):
        # plot a line around 0
        ax[0, i].axhline(0, color='black', ls='--', zorder=3)
        ax[1, i].axhline(0, color='black', ls='--', zorder=3)

        relative_alpha = results_sim_study[f'{selection_data}_{selection_inference}_alpha']['relative_bias_median'][i]
        relative_omicron = results_sim_study[f'{selection_data}_{selection_inference}_omicron']['relative_bias_median'][i]

        # Define the error as [lower_error, upper_error]
        lower_quantile, upper_quantile = np.quantile(relative_alpha, [0.025, 0.975])
        yerr_alpha = np.array([[np.median(relative_alpha) - lower_quantile],
                               [upper_quantile - np.median(relative_alpha)]])
        lower_quantile, upper_quantile = np.quantile(relative_omicron, [0.025, 0.975])
        yerr_omicron = np.array([[np.median(relative_omicron) - lower_quantile],
                                 [upper_quantile - np.median(relative_omicron)]])


        handle_alpha_mean = ax[0, i].errorbar(
            j,
            np.median(relative_alpha),
            yerr=yerr_alpha,
            color=colors[j // 2], marker='x', label=r'Median Bias', zorder=4
        )
        handle_omicron_mean = ax[1, i].errorbar(
            j,
            np.median(relative_omicron),
            yerr=yerr_omicron,
            color=colors[j // 2], marker='x', label=r'Median Bias', zorder=4
        )
        ax[0, i].set_title(param_names[index_i])
        #ax[1, i].set_xlim(-0.5, 3.5)
        ax[1, i].set_xlim(-1.5, 4.5)
        ax[1, i].set_ylim(-1, 4)
        #ax[1, i].set_xticks([0, 1, 2, 3], labels=['PedCov', 'Random', 'PedCov', 'Random'], rotation=60)
        ax[1, i].set_xticks([0, 3], labels=['PedCov', 'Random'], rotation=60)
        ax[0, 0].set_ylabel('Median Relative Bias \n Alpha')
        ax[1, 0].set_ylabel('Median Relative Bias \n Omicron')

color_patch = [Patch(color=colors[i], label=f'Data generated with {["PedCov", "Random"][i]}') for i in range(2)]
leg = fig.legend(handles=color_patch, bbox_to_anchor=(0.5, -0.05), loc='lower center', ncol=2)
#fig.savefig(f'{results_folder}/{amortizer_name}_sim_study_data_bias.png', bbox_inches='tight')
plt.show()

# Apply trained model to real data

In [ ]:
# specify the PedCov path
data_path_alpha = 'Simulator/pedcovid_data_structure_alpha.txt'  # todo: exchange with real PedCov of alpha
data_path_omicron = 'Simulator/pedcovid_data_structure_omicron.txt'  # todo: exchange with real PedCov of omicron
os.makedirs(results_folder+'/real PedCov', exist_ok=True)

community_infection = {  # must be one of the predefined values, otherwise networks are unreliable
    'alpha': 0.002,
    'omicron': 0.02
}

In [ ]:
# load the PedCov
dfs = {
    'alpha': pd.read_csv(data_path_alpha, delimiter=' ', index_col=0),
    'omicron': pd.read_csv(data_path_omicron, delimiter=' ', index_col=0)
}
prior_samples = renormalize_params(prior(1000))

real_data = {
    'alpha': None,
    'omicron': None
}

# prepare the PedCov for neural networks
for variant in ['alpha', 'omicron']:
    household_data = normalize_household_data(dfs[variant], minimal_length=9)[np.newaxis]
    real_data[variant] = {
        'sim_data': household_data,
        'sim_non_batchable_context': variant,
        'sim_batchable_context': ['pedcov', community_infection[variant]]
    }
    real_data[variant].update({'configured_data': trainer.configurator(real_data[variant])})

In [ ]:
# get posterior samples
for variant in ['alpha', 'omicron']:
    posterior_samples_real = (
        trainer.amortizer.sample(real_data[variant]['configured_data'], n_samples=prior_samples.shape[0])
    )

    # save to csv
    #pd.DataFrame(real_data[variant]['posterior_samples'], columns=param_names).to_csv(
    #    f'{results_folder}/real PedCov/posterior_samples_{variant}.csv'
    #)


In [ ]:
# plot posterior samples vs prior
for variant in ['alpha', 'omicron']:
    print(f"Variant: {variant}")
    fig = bf.diagnostics.plot_posterior_2d(
        posterior_draws=real_data[variant]['posterior_samples'],
        prior_draws=prior_samples,
        param_names=param_names,
        label_fontsize=22,
        legend_fontsize=24
    )
    ax = fig.get_axes()
    for i, a in enumerate(ax):
        # plot only on the diagonal
        if i == i // len(param_names) * (len(param_names)+1):
            a.axvline(1, color='b')
    #plt.savefig(f'{results_folder}/real PedCov/posterior_{variant}.png', bbox_inches='tight')
    plt.show()

In [ ]:
# plot credible intervals for each parameter and each variant
for variant in ['alpha', 'omicron']:
    ax = sampling_parameter_cis(real_data[variant]['posterior_samples'], alpha=[99, 95, 80],
                                param_names=param_names, title=f"Real Data Posterior CIs - {variant}")
    # add vertical line at 1 for the mu parameters
    ax.vlines(1, ymin=1.75, ymax=8.25, color='grey', linestyle='--')
    #plt.savefig(f'{results_folder}/real PedCov/CIs_{variant}.png', bbox_inches='tight')
    plt.show()